# 16 · 프리스트레스트 콘크리트 — KDS 14 20 60

원 문서의 `prestressed_section.ipynb` 에 대응한다.

| 항목 | 조문 |
|---|---|
| 긴장재 허용응력 $\min(0.80f_{pu}, 0.94f_{py})$ 등 | KDS 14 20 60 4.2.2 |
| 콘크리트 허용응력, 균열등급 U/T/C | KDS 14 20 60 4.2.1, 4.2.2 |
| 마찰 손실 $P_{px}=P_{pj}e^{-(Kl_{px}+\mu_p\alpha_{px})}$ | KDS 14 20 60 4.3 |
| 부착 긴장재 $f_{ps}$ | KDS 14 20 60 4.4.2(3) |
| PSC 변형률한계 0.002 / 0.005 | KDS 14 20 20 4.1.2(3), (4) |

In [1]:
import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [2]:
from concreteproperties import (
    PrestressedSection,
    SteelStrand,
    StrandHardening,
    add_bar_rectangular_array,
)
from sectionproperties.pre.library import rectangular_section

from concreteproperties_kds import KDS
from concreteproperties_kds.psc import (
    KDSPrestressed,
    PrestressLosses,
    allowable_concrete_stress_service,
    allowable_concrete_stress_transfer,
    allowable_tendon_stress,
    anchorage_set_loss,
    capacity_reduction_factor_psc,
    creep_loss,
    elastic_shortening_loss,
    friction_loss,
    relaxation_loss,
    shrinkage_loss,
    tendon_stress_bonded,
    tendon_stress_unbonded,
)

FPU, E_P = 1860.0, 200e3      # Eps = 200,000 MPa (KDS 14 20 10 식 4.3-6)
FPY = 0.9 * FPU               # 저릴랙세이션
FCK, FCI = 40.0, 30.0
N_STRAND, A_STRAND, SPAN = 4, 138.7, 20000.0

## 허용응력 (KDS 14 20 60 4.2)

In [3]:
print(f"긴장 중        = {allowable_tendon_stress(FPU, FPY, 'jacking'):8.1f} MPa"
      f"   min(0.80fpu, 0.94fpy)")
print(f"정착 직후      = {allowable_tendon_stress(FPU, FPY, 'anchorage'):8.1f} MPa"
      f"   min(0.74fpu, 0.82fpy)")
print(f"정착장치       = "
      f"{allowable_tendon_stress(FPU, FPY, 'anchorage_device'):8.1f} MPa"
      f"   0.70fpu")
print()

c_t, t_t = allowable_concrete_stress_transfer(fci=FCI)
c_s, t_s = allowable_concrete_stress_service(fck=FCK, crack_class="U")
c_sus, _ = allowable_concrete_stress_service(fck=FCK, sustained=True)

print(f"도입 직후 콘크리트 = {c_t:7.2f} / {t_t:7.2f} MPa  (압축/인장)")
print(f"사용 전체하중      = {c_s:7.2f} / {t_s:7.2f} MPa  (비균열등급 U)")
print(f"사용 지속하중 압축 = {c_sus:7.2f} MPa")

긴장 중        =   1488.0 MPa   min(0.80fpu, 0.94fpy)
정착 직후      =   1372.7 MPa   min(0.74fpu, 0.82fpy)
정착장치       =   1302.0 MPa   0.70fpu

도입 직후 콘크리트 =   18.00 /   -1.37 MPa  (압축/인장)
사용 전체하중      =   24.00 /   -3.98 MPa  (비균열등급 U)
사용 지속하중 압축 =   18.00 MPa


## 프리스트레스 손실 (KDS 14 20 60 4.3)

In [4]:
kds = KDS()
conc = kds.create_concrete_material(compressive_strength=FCK)
conc_i = kds.create_concrete_material(compressive_strength=FCI)

f_pj = 0.75 * FPU
a_ps = N_STRAND * A_STRAND

_, friction_force = friction_loss(
    p_pj=f_pj * a_ps, mu_p=0.20, alpha_px=0.15,
    k_wobble=6.6e-7, l_px=SPAN / 2,
)

losses = PrestressLosses(
    f_pj=f_pj,
    friction=friction_force / a_ps,
    anchorage=anchorage_set_loss(slip=6.0, e_p=E_P, length=SPAN / 2),
    elastic=elastic_shortening_loss(
        f_cgp=8.0, e_p=E_P, e_ci=conc_i.elastic_modulus,
        post_tensioned=True, n_tendons=N_STRAND,
    ),
    creep=creep_loss(
        f_cgp=8.0, e_p=E_P, e_c=conc.elastic_modulus,
        creep_coefficient=2.0,
    ),
    shrinkage=shrinkage_loss(e_p=E_P, eps_sh=300e-6),
    relaxation=relaxation_loss(f_pi=0.70 * FPU, fpy=FPY),
)
losses.print_results()

프리스트레스 손실 (KDS 14 20 60 4.3)
잭킹 응력          fpj    =    1395.00 MPa
--------------------------------------------------------
마찰                      =      50.13 MPa
정착장치 활동             =     120.00 MPa
탄성변형                  =      21.79 MPa
  즉시 손실 소계          =     191.92 MPa
--------------------------------------------------------
크리프                    =     106.64 MPa
건조수축                  =      60.00 MPa
릴랙세이션                =      28.87 MPa
  시간적 손실 소계        =     195.51 MPa
--------------------------------------------------------
전체 손실                 =     387.43 MPa
손실률                    =      27.77 %
유효 프리스트레스  fpe    =    1007.57 MPa


In [5]:
labels = ["friction", "anchorage", "elastic", "creep", "shrinkage", "relax."]
values = [
    losses.friction, losses.anchorage, losses.elastic,
    losses.creep, losses.shrinkage, losses.relaxation,
]

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.bar(labels, values)
ax.set_ylabel("stress loss (MPa)")
ax.set_title(
    f"Prestress losses (total {losses.total:.0f} MPa, "
    f"{losses.loss_ratio * 100:.1f} %)"
)
ax.grid(alpha=0.3, axis="y")

## 긴장재의 극한 응력 (KDS 14 20 60 4.4.2)

In [6]:
d_p, b = 680.0, 400.0
rho_p = a_ps / (b * d_p)

print(f"긴장재비  rho_p = {rho_p:.5f}")
print(f"부착   fps = {tendon_stress_bonded(FPU, FCK, rho_p, 0.28):8.1f} MPa")
print(f"비부착 fps = "
      f"{tendon_stress_unbonded(losses.f_pe, FCK, rho_p, FPY, SPAN / 800):8.1f} MPa")
print(f"유효 프리스트레스 fpe = {losses.f_pe:8.1f} MPa")

긴장재비  rho_p = 0.00204
부착   fps =   1798.3 MPa
비부착 fps =   1273.7 MPa
유효 프리스트레스 fpe =   1007.6 MPa


## 단면 해석

In [7]:
strand = SteelStrand(
    name="SWPC 7B 15.2mm",
    density=7.85e-6,
    stress_strain_profile=StrandHardening(
        yield_strength=FPY, elastic_modulus=E_P,
        fracture_strain=0.035, breaking_strength=FPU,
    ),
    colour="slategrey",
    prestress_stress=losses.f_pe,
)

geom = rectangular_section(d=800, b=400, material=conc)
geom = add_bar_rectangular_array(
    geometry=geom, area=A_STRAND, material=strand,
    n_x=N_STRAND, x_s=80, anchor=(80, 120), n=8,
)
ps_sec = PrestressedSection(geom)
ps_sec.plot_section()

/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 53080 (\N{HANGUL SYLLABLE KON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 53356 (\N{HANGUL SYLLABLE KEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 47532 (\N{HANGUL SYLLABLE RI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 53944 (\N{HANGUL SYLLABL

<Axes: title={'center': 'Reinforced Concrete Section'}>

In [8]:
kds_ps = KDSPrestressed(column_type="tie")
kds_ps.assign_prestressed_section(ps_sec)

f_res, u_res, phi = kds_ps.ultimate_bending_capacity(positive=True)
eps_t = kds_ps.net_tensile_strain(theta=0, d_n=u_res.d_n)

print(f"중립축 깊이   c      = {u_res.d_n:8.2f} mm")
print(f"순인장변형률  et     = {eps_t:8.5f}")
print(f"강도감소계수  phi    = {phi:8.3f}")
print(f"공칭 휨강도   Mn     = {u_res.m_x / 1e6:8.2f} kN.m")
print(f"설계 휨강도   phi*Mn = {f_res.m_x / 1e6:8.2f} kN.m")

중립축 깊이   c      =    91.71 mm
순인장변형률  et     =  0.02117
강도감소계수  phi    =    0.850
공칭 휨강도   Mn     =   641.93 kN.m
설계 휨강도   phi*Mn =   545.64 kN.m


## PSC 부재의 강도감소계수

프리스트레스트 부재는 변형률한계가 $f_y$ 에 의존하지 않고 **0.002 / 0.005
고정값**이다 (KDS 14 20 20 4.1.2(3), (4)).

In [9]:
eps = np.linspace(0, 0.008, 300)
fig, ax = plt.subplots(figsize=(6.5, 4))
for ctype, label in [("tie", "tie (0.65)"), ("spiral", "spiral (0.70)")]:
    ax.plot(
        eps,
        [
            capacity_reduction_factor_psc(eps_t=float(e), column_type=ctype)
            for e in eps
        ],
        label=label,
    )
ax.axvline(0.002, ls=":", color="grey", lw=0.8)
ax.axvline(0.005, ls=":", color="grey", lw=0.8)
ax.set_xlabel("net tensile strain, eps_t")
ax.set_ylabel("phi")
ax.set_title("Strength reduction factor, prestressed members")
ax.legend()
ax.grid(alpha=0.3)